# Building from scratch

Basic building block

In [1]:
import torch
import torch.nn as nn
class ConvBNReLU(nn.Module):
  def __init__(self,in_c,out_c,k,s=1,p=2):
    super().__init__()
    self.block=nn.Sequential(
        nn.Conv2d(in_channels=in_c,out_channels=out_c,kernel_size=k,stride=s,padding=p,bias=False),
        #bias =false ensures that conv layer will not learn any bias term cuz batch norm introduces its own learnable bias(beta params) and
        #scale(gamma params) that absorb any bias from the preceding layer.
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True)
    )
  def forward(self,x):
    return self.block(x)

Factorized 5x5

In [2]:
class Factorized5x5(nn.Module):
  def __init__(self,in_c,out_c):
    super().__init__()
    self.block=nn.Sequential(
        ConvBNReLU(in_c,out_c,3,p=1),
        ConvBNReLU(out_c,out_c,3,p=1)
    )
  def forward(self,x):
    return self.block(x)

Inception v2 block

In [3]:
class InceptionV2Block(nn.Module):
  def __init__(self,in_c):
    super().__init__()

    #1x1
    self.branch1=ConvBNReLU(in_c,64,1)

    #1x1 -> 3x3
    self.branch2=nn.Sequential(
        ConvBNReLU(in_c,48,1),
        ConvBNReLU(48,64,3,p=1)
    )
    #1x1 -> (3x3 + 3x3) instead of 5x5
    self.branch3=nn.Sequential(
        ConvBNReLU(in_c,64,1),
        Factorized5x5(64,96)
    )
    #pooling branch
    self.branch4=nn.Sequential(
        nn.MaxPool2d(kernel_size=3,stride=1,padding=1),
        ConvBNReLU(in_c,32,1)
    )
  def forward(self,x):
    b1=self.branch1(x)
    b2=self.branch2(x)
    b3=self.branch3(x)
    b4=self.branch4(x)
    return torch.cat([b1,b2,b3,b4], dim=1)

In [4]:
x = torch.randn(1, 192, 28, 28)
model = InceptionV2Block(192)

out = model(x)
print(out.shape)

torch.Size([1, 256, 32, 32])
